# pick
Original dataset annotations; validation selects the checkpoint.


## 1. Environment
On Kaggle, attach `vhngthnh/reconia-data` and enable the `GH_TOKEN` secret. Locally, set `RECONIA_DATA_DIR` to the raw dataset root. Existing source and outputs are preserved.
Choose `DATASET`: `sroie`, `cord_v2`, or `wildreceipt`.


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

DATASET = "sroie"
GIT_REF = "codex/original-data-six-models"
repo_dir = Path("/kaggle/working/reconia") if Path("/kaggle").exists() else Path.cwd()
if repo_dir.name == "notebooks":
    repo_dir = repo_dir.parent
if not repo_dir.exists():
    import base64
    from kaggle_secrets import UserSecretsClient
    env = os.environ.copy()
    token = UserSecretsClient().get_secret("GH_TOKEN")
    auth = base64.b64encode(("x-access-token:" + token).encode()).decode()
    env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
               GIT_CONFIG_VALUE_0="Authorization: Basic " + auth)
    subprocess.run(["git", "clone", "--depth", "1", "--branch", GIT_REF,
                    "https://github.com/tamir39/reconia.git", str(repo_dir)], env=env, check=True)
    del env, token, auth
os.chdir(repo_dir)
sys.path.insert(0, str(repo_dir))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Source:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())


## 2. Prepare and inspect the selected dataset


In [ ]:
from importlib import import_module
from utils.common import get_cache_dir, get_data_dir

print("Raw data:", get_data_dir(DATASET))
prepare = getattr(import_module(f"prepare.prepare_{DATASET}"), f"prepare_{DATASET}")
prepare()
print("Prepared:", get_cache_dir(DATASET))


In [ ]:
import json
from IPython.display import Image, display

with (get_cache_dir(DATASET) / "train.jsonl").open(encoding="utf-8") as file:
    sample = json.loads(file.readline())
image_path = get_data_dir(DATASET) / sample["image_path"]
print("Sample:", sample["id"], "| Image:", image_path)
display(Image(filename=str(image_path), width=350))
display({"text_box_excerpt": sample["ocr"][:5], "target_gt": sample["gt"]})


## 3. Pretrained weights
PICK initializes its text/graph/decoder layers from scratch. The image encoder loads torchvision ResNet-50 pretrained weights when the model is created; allow Internet access or cache these weights first.


In [ ]:
from torchvision.models import ResNet50_Weights

print("Image backbone:", ResNet50_Weights.DEFAULT.url)


## 4. Train and validate
Precision, Recall and F1 use exact field-value matches after lowercasing and whitespace normalization, averaged across fields (Macro). Validation runs after each epoch; its Macro-F1 selects the best checkpoint.


In [ ]:
from models.pick.train import train_model

train_model(DATASET, epochs=30, batch_size=4, accum_steps=2,
            lr=0.0001, min_epochs=5, patience=5, resume=False)


## 5. Validation history


In [ ]:
import pandas as pd

display(pd.read_csv(repo_dir / "results" / DATASET / "pick" / "history.csv"))


## 6. Final test (run after experiments are fixed)
Loads the best validation checkpoint; does not update weights.


In [ ]:
RUN_TEST = False
if RUN_TEST:
    train_model(DATASET, batch_size=4, test_only=True)
